# CoT1 copy.ipynb

Original Bi-CoT research notebook, recovered from the author’s AI_Study archive. Outputs, execution state, and machine metadata were removed; API credentials now come from `OPENAI_API_KEY`, and local paths now use `../data/workspace` relative to this notebook directory. See `docs/WORKFLOW.md` before selecting cells. Cells include experimental alternatives and data writes; this is not a single Run All pipeline.


아래는 문서 반환받는 코드

In [ ]:
import json
from typing import List, Dict, Set

def extract_unique_title_sents_from_jsons(q_json_paths: List[str]) -> List[Dict[str, str]]:
    """
    여러 q.json 파일 경로를 받아, 모든 candidate_chains 내에서
    중복되지 않는 {"title": title, "sents": paragraph} 리스트를 반환
    """
    seen_paragraphs: Set[str] = set()
    result: List[Dict[str, str]] = []

    for q_json_path in q_json_paths:
        with open(q_json_path, "r", encoding="utf-8") as f:
            q_data = json.load(f)

        candidate_chains = q_data.get("candidate_chains", [])

        for chain in candidate_chains:
            for doc in chain:
                title = doc.get("title", "")
                sents = doc.get("sents", [])
                paragraph = " ".join(sents).strip()

                if paragraph and paragraph not in seen_paragraphs:
                    result.append({"title": title, "sents": paragraph})
                    seen_paragraphs.add(paragraph)

    return result

아래는 메인 순방향 CoT

In [ ]:
import os
import json
import re
from typing import List, Dict, Set
from openai import OpenAI

client = OpenAI(api_key=__import__("os").environ["OPENAI_API_KEY"])  # 본인 API 키 사용

def merge_context_details(context_details_list: List[List[Dict[str, str]]]) -> List[Dict[str, str]]:
    seen_sents: Set[str] = set()
    result: List[Dict[str, str]] = []
    for context_details in context_details_list:
        for item in context_details:
            sents = item["sents"]
            if sents not in seen_sents:
                result.append(item)
                seen_sents.add(sents)
    return result
import json

def extract_exist_json_line(s):
    lines = s.strip().split('\n')
    for line in reversed(lines):  # 마지막 줄부터 탐색
        line_stripped = line.strip()
        # JSON 포맷이고 "exist"가 key로 포함
        if line_stripped.startswith('{') and '"exist"' in line_stripped:
            try:
                obj = json.loads(line_stripped)
                exist_val = str(obj.get("exist", "")).lower()
                if exist_val in ["yes", "no"]:
                    return obj
            except Exception:
                continue
    return None


def extract_answer_candidates(*answer_jsons):
    answers = []
    for a in answer_jsons:
        ans_field = a.get("answer", "")
        if isinstance(ans_field, dict):
            ans = str(ans_field.get("answer", "")).strip()
        else:
            ans = str(ans_field).strip()
        if ans:
            answers.append(ans)
    return list(set(answers))  # 중복제거

def call_openai(prompt, model="gpt-4o", max_tokens=1024, temperature=0):
    #print("\n======================")
    #print("💬 [OpenAI Prompt]")
    #print(prompt)
    print("======================")
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    output = response.choices[0].message.content.strip()
    print("📝 [OpenAI Response]")
    print(output)
    print("======================\n")
    return output

def find_final_answer_and_evidence_via_api(question: str, context_items: List[Dict[str, str]], answer_candidates: List[str]) -> (str, str):
    # context는 번호+타이틀+문장으로 구성
    context_str = "\n".join(
        [f"{i+1}. {item['title']}: {item['sents']}" for i, item in enumerate(context_items)]
    )

    # 답 후보를 한 줄로
    candidate_str = "; ".join(answer_candidates)

    # API 프롬프트 설계
    question_block = f"Question: {question}\nAnswer candidates: {candidate_str}\n\nContext:\n{context_str}\n\n"
    prompt_preamble = prompt_preamble = prompt_preamble = """You are a step-by-step reasoning assistant. Read the context and reason carefully.

Rules:
- Do NOT restate the question, candidates, or context in your answer. Start directly with your reasoning.
- Think step-by-step:
  1) Extract the constraints from the question (what must be true).
  2) Scan each context passage in order; note whether it contains the required information.
  3) For each candidate, check against the context with the constraints.
  4) Decide if a candidate fits exactly; otherwise derive a better answer strictly from the context.
- Output must be ONE of the following JSON formats (and nothing else):
  - {"exist": "Yes", "answer": 1, "evidence": <num>}
  - {"exist": "Yes", "answer": 2, "evidence": <num>}
  - {"exist": "No", "answer": "<new_answer>", "evidence": <num>}
  - {"exist": "No", "answer": "No answer found", "evidence": "No evidence found"}

Definitions:
- If neither candidate is correct but the correct answer text appears in the context, copy that exact span as <new_answer>.
- If the exact span does not appear but the answer can be *utilized* (i.e., minimally composed/normalized only from words present in the context without adding new tokens), you may output that composed <new_answer>. In all cases, cite the single most decisive passage number as evidence.
- If no recoverable answer exists from the context (neither exact span nor a minimal composition), output:
  {"exist": "No", "answer": "No answer found", "evidence": "No evidence found"}
- "evidence": <num> denotes the single most decisive Context passage that supports your final answer.
- <num> must be an integer and must equal the 1-based index of a passage in the provided Context
- Use exactly one number. If no supporting passage exists, output "evidence": "No evidence found".

[Example]

Question: Which city hosted the Olympic Games in 1998?
Answer candidates: Nagano; Sydney

Context:
1. Olympics: The Olympics are a large international sports festival. The 2021 Summer Olympics were held in Tokyo.
2. Wintersports in Japan: Nagano is a city in Japan. In 1998, the Winter Olympics were held in Nagano.
3. Sydney: Sydney is a city in Australia. In 1998, an international music festival was held there.

Chain-of-Thought Reasoning:
- Constraints: The answer must be a city that hosted the Olympic Games in 1998.
- Context 1: Mentions Olympics and 2021 Tokyo → does NOT satisfy the 1998 host constraint.
- Context 2: Contains the sentence “In 1998, the Winter Olympics were held in Nagano.” This sentence directly satisfies the constraints and names the city (Nagano).
- Context 3: Mentions a music festival in 1998, not the Olympics → irrelevant for the constraints.
- Candidate check: Candidate 1 = "Nagano" matches the city in Context 2; Candidate 2 = "Sydney" does not match.

Final Output:
{"exist": "Yes", "answer": 1, "evidence": 2}


<Rules>
You must first produce a “Chain-of-Thought Reasoning” section that (a) extracts the constraints from the question, (b) evaluates each Context passage by number against those constraints, and (c) tests each candidate accordingly; only after this step-by-step reasoning, on a new line, output exactly one JSON object in the required format—any answer that skips or shortens the reasoning or outputs JSON without it is invalid.


=== BEGIN TASK ===

"""

    prompt = prompt_preamble + "\n" + question_block

    out = call_openai(prompt, model="gpt-4o", max_tokens=1024, temperature=0)
    res = extract_exist_json_line(out)

    # 5) answer 결정 + evidence_title 매핑
    answer = "No answer found"
    evidence_title = "No evidence found"

    if res:
        exist = str(res.get("exist", "")).lower()
        ans_field = res.get("answer", "")

        # exist 처리
        if exist == "yes":
            # 정답 후보 인덱스 → 실제 텍스트
            try:
                idx = int(ans_field)
                if 1 <= idx <= len(answer_candidates):
                    answer = answer_candidates[idx - 1]
                else:
                    answer = ""
            except Exception:
                answer = ""
        elif exist == "no":
            # 새로 생성한 답(또는 No answer found)
            answer = str(ans_field)

        # evidence 처리
        evidence_val = res.get("evidence", None)
        if isinstance(evidence_val, int):
            if 1 <= evidence_val <= len(context_items):
                evidence_title = context_items[evidence_val - 1].get("title", "")
            else:
                evidence_title = "No evidence found"
        elif isinstance(evidence_val, str) and evidence_val.strip().lower() == "no evidence found":
            evidence_title = "No evidence found"
        else:
            # 타입이 이상하면 안전하게 처리
            evidence_title = "No evidence found"

    return answer, evidence_title
def save_cot2_a_json(save_path: str, question: str, answer: str, evidence_title: str):
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    answer_obj = {
        "question": question,
        "answer": answer,
        "evidence_title": evidence_title
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(answer_obj, f, ensure_ascii=False, indent=2)

def main(example_id_dir: str):
    # context_detail 모으기
    context_details_list = []
    for path in [
        os.path.join(example_id_dir, "0", "a.json"),
        os.path.join(example_id_dir, "3", "0", "a.json"),
        os.path.join(example_id_dir, "3", "1", "a.json"),
    ]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
            if "context_detail" in data:
                context_details_list.append(data["context_detail"])
    merged_context = merge_context_details(context_details_list)
    if not merged_context:
        print(f"❌ context가 없음. 종료.")
        return

    # 1/0 question
    q_path = os.path.join(example_id_dir, "3", "0", "a.json")
    if not os.path.exists(q_path):
        print(f"❌ {q_path} 없음. 종료.")
        return
    with open(q_path, "r", encoding="utf-8") as f:
        q_data = json.load(f)
    question = q_data.get("question", "")

    # 1/0, 1/1 answer 후보
    ans_path_1 = os.path.join(example_id_dir, "3", "0", "a.json")
    ans_path_2 = os.path.join(example_id_dir, "3", "1", "a.json")
    answer_jsons = []
    for path in [ans_path_1, ans_path_2]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                answer_jsons.append(json.load(f))
    answer_candidates = extract_answer_candidates(*answer_jsons)
    if not answer_candidates:
        answer_candidates = []

    # OpenAI API를 이용해 최종 answer, evidence_title 뽑기
    answer, evidence_title = find_final_answer_and_evidence_via_api(question, merged_context, answer_candidates)

    # cot2/a.json로 저장
    save_path = os.path.join(example_id_dir, "3","cot2", "a.json")
    save_cot2_a_json(save_path, question, answer, evidence_title)
    print(f"CoT 결과 저장됨: {save_path}")
    print(f"question: {question}\nanswer: {answer}\nevidence_title: {evidence_title}")

# 사용 예시 (폴더 단위 반복)
def run_all(base_dir):
    example_ids = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    for example_id in example_ids:
        main(os.path.join(base_dir, example_id))

def run_one(base_dir, example_id):
    example_id_dir = os.path.join(base_dir, example_id)
    if os.path.isdir(example_id_dir):
        main(example_id_dir)
    else:
        print(f"{example_id_dir} does not exist.")


In [ ]:
base_dir = "../data/workspace/Last_CoT_123"
run_all(base_dir)


#example_id = ['5a89fc305542993b751ca9dc', '5abbdb9a55429931dba145ad', '5a81d6cd55429903bc27b9e0', '5abb66c05542992ccd8e7f3e', '5ae7f7d85542994a481bbe52', '5a8c6bd55542995e66a475ea', '5abe45b855429976d4830ad0', '5ab3eac5554299753aec59ec', '5ab82fe155429919ba4e225a', '5adc02445542994650320c4e', '5ae1f70c5542997f29b3c1c5', '5a82bb3c55429966c78a6a86', '5a8f8c6d5542997ba9cb32b1', '5ae3e5d05542994393b9e79f', '5a8595a85542992a431d1b48', '5ac02c4a5542992a796decbd', '5a75ac415542992d0ec05fdb', '5a8051265542992bc0c4a6f8', '5a728b1c5542992359bc30e0', '5a7744555542994aec3b725b', '5a8b5c8155429949d91db56d', '5ae23d945542996483e64939', '5a79f95d5542996c55b2dcbd', '5a9070dd5542995b442420b5', '5a79caec5542994bb94570ad', '5a87279b5542991e771816f7', '5a8a416955429930ff3c0d2b', '5a8753c05542994846c1cd62', '5ac4241e5542997ea680ca10', '5a79186f55429907847277af', '5ab80faf55429916710eafd9', '5a76fe7b5542994aec3b71b2', '5ac4e13f554299076e296e2d', '5ae22670554299234fd043fc', '5ab661ac5542995eadeeffb1', '5a75176a5542993748c897b6', '5ae7ad385542993210983ef0', '5abbd5a75542993f40c73bd3', '5ae7257a554299572ea54731', '5adca8215542994ed6169bbc', '5a83161555429940e5e1a958', '5ab6c63755429953192ad38b', '5ae601e45542992663a4f23b', '5abd96fe5542992ac4f382d9', '5abd27ba5542993062266ba7', '5ab74eb85542993667793fb8', '5a81736255429903bc27b96e', '5a8a0ed55542992e4fca84e8', '5ab43b9d5542990594ba9bc3', '5a7666095542992d0ec060db', '5ae40a2955429970de88d8a3', '5a7d9d2b5542997cc2c47464', '5a88f2cc55429938390d3ffb', '5ab204795542993be8fa98ad', '5ae1e4095542997283cd22bf', '5ae7620f5542997ec2727627', '5ab844815542992aa3b8c8af', '5adde5155542997dc7907092', '5a72d7b35542991f9a20c5be', '5a72814f5542994cef4bc2eb', '5adfa97555429942ec259ae1', '5a7130365542994082a3e645', '5a83172455429954d2e2ec14']
#example_id = ['5a89761d5542995153361310']
#for i in example_id:
    #run_one(base_dir, i)

아래는 단순 판단이 필요한 경우만 사용한 것  (3번 comparison)

In [ ]:
import os
import json
import re
from typing import Dict
from openai import OpenAI

BASE_DIR = "../data/workspace/Last_CoT_123"

client = OpenAI(api_key=__import__("os").environ["OPENAI_API_KEY"])

def call_openai(prompt, model="gpt-4o", max_tokens=512, temperature=0):
    print("======================")
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature,
    )
    output = response.choices[0].message.content.strip()
    print("📝 [OpenAI Response]")
    print(output)
    print("======================\n")
    return output

def extract_json_obj(s: str) -> dict | None:
    """응답에서 JSON 오브젝트 한 개만 안전하게 뽑아온다."""
    # 먼저 전체를 시도
    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    # 라인/블록 중 마지막 JSON만 파싱
    matches = re.findall(r'\{.*?\}', s, flags=re.DOTALL)
    for m in reversed(matches):
        try:
            obj = json.loads(m)
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue
    return None

PROMPT_TEMPLATE = (
    "You are a strict judge of whether a question can be answered WITHOUT any external context "
    "(i.e., only with commonsense, arithmetic, string/number comparison, or trivial logic).\n"
    "If it is self-contained and answerable now, provide the shortest correct answer.\n"
    "If it clearly requires additional information or non-trivial world knowledge, mark it unsolvable.\n\n"
    "Question: {question}\n\n"
    "Output ONLY one JSON object:\n"
    '- If solvable: {{"solvable": true, "answer": "<short answer>"}}\n'
    '- If not solvable: {{"solvable": false, "answer": ""}}\n'
    "No explanations, no extra text.\n"
)

def can_answer_now(question: str) -> tuple[bool, str]:
    prompt = PROMPT_TEMPLATE.format(question=question)
    out = call_openai(prompt, model="gpt-4o", max_tokens=128, temperature=0)
    obj = extract_json_obj(out)
    if not obj:
        return (False, "")
    solvable = bool(obj.get("solvable", False))
    ans = str(obj.get("answer", "")).strip()
    if solvable and not ans:
        # solvable인데 빈답이면 실패 취급
        return (False, "")
    return (solvable, ans)

def load_questions(base_dir: str) -> Dict[str, str]:
    """각 id 폴더의 3/0/a.json에서 question을 수집하여 {id: question} 반환"""
    q_by_id: Dict[str, str] = {}
    ids = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    for ex_id in sorted(ids):
        a_path = os.path.join(base_dir, ex_id, "3", "0", "a.json")
        if not os.path.exists(a_path):
            continue
        try:
            with open(a_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"[WARN] JSON load failed: {a_path} ({e})")
            continue
        q = str(data.get("question", "")).strip()
        if q:
            q_by_id[ex_id] = q
    return q_by_id

def main():
    # 1) 질문 수집
    id2question = load_questions(BASE_DIR)
    print(f"Collected questions: {len(id2question)}")

    # 2) 상식/비교로 바로 답할 수 있는 경우만 해결
    id2answer: Dict[str, str] = {}
    for ex_id, q in id2question.items():
        solvable, ans = can_answer_now(q)
        if solvable:
            id2answer[ex_id] = ans  # 답 사전 추가
        # solvable이 False면 스킵 (요청사항)

    # 3) 결과 출력
    print("\n=== id -> question (sample up to 10) ===")
    for i, (k, v) in enumerate(id2question.items()):
        print(f"{k}: {v}")

    print(f"\nTotal questions: {len(id2question)}")
    print(f"Solved without context: {len(id2answer)}")

    print("\n=== id -> answer (self-contained solved) ===")
    for k, v in id2answer.items():
        print(f"{k}: {v}")

    # (선택) 파일로 저장하고 싶으면 주석 해제
    # with open(os.path.join(BASE_DIR, "id2question.json"), "w", encoding="utf-8") as f:
    #     json.dump(id2question, f, ensure_ascii=False, indent=2)
    with open(os.path.join(BASE_DIR, "id2answer_self_contained.json"), "w", encoding="utf-8") as f:
         json.dump(id2answer, f, ensure_ascii=False, indent=2)

if __name__ == "__main__":
    main()